# Homework 3 BAN 5600
## Google Sheets + Gemini API Case Study

Name: Ryan Cooper  
Course: BAN 5600  
Professor: Omar El Mghari  

This notebook shows how to:
1. Authenticate Google Colab
2. Connect Python to Google Sheets
3. Create a spreadsheet
4. Upload sample data
5. Use Gemini API to interpret the data
6. Show the results
7. Add a creative supply chain example

In [24]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user so Colab can access Google services
auth.authenticate_user()

# Get credentials from the Google account signed into Colab
creds, _ = default()

# Authorize gspread so Python can work with Google Sheets
gc = gspread.authorize(creds)

print("Successfully authenticated!")

Successfully authenticated!


In [25]:
# Create a new spreadsheet
sh = gc.create("Commission Report")

# Get the first worksheet
worksheet = sh.get_worksheet(0)

# Add the header row
headers = ["Sales Amount", "Region", "Final Commission"]
worksheet.append_row(headers)

print("Header row added successfully!")
print("Spreadsheet URL:", sh.url)

Header row added successfully!
Spreadsheet URL: https://docs.google.com/spreadsheets/d/1SmR_jiVyp1Xf4WcUtjwYLnPdSQ-nLaeS14Pr2LTa8e8


In [26]:
# Sample data from the workflow example
sales_data = [
    [1200.00, "E", 98.00],
    [5500.00, "W", 355.00],
    [800.00, "C", 16.00]
]

print("Writing original sample data to Google Sheets...")

# Loop through each row and append it to the worksheet
for row in sales_data:
    worksheet.append_row(row)

print("Done! Original sales sheet created.")
print("Sheet URL:", sh.url)

Writing original sample data to Google Sheets...
Done! Original sales sheet created.
Sheet URL: https://docs.google.com/spreadsheets/d/1SmR_jiVyp1Xf4WcUtjwYLnPdSQ-nLaeS14Pr2LTa8e8


In [27]:
# Create a second spreadsheet with my own supply chain example
shipment_report = gc.create("Supply Chain Delay Report")
shipment_sheet = shipment_report.get_worksheet(0)

# Add headers for the supply chain example
shipment_headers = ["Order ID", "Region", "Delay Days"]
shipment_sheet.append_row(shipment_headers)

# Sample supply chain data
shipment_data = [
    [1001, "East", 2],
    [1002, "West", 5],
    [1003, "Central", 1]
]

print("Writing supply chain data to Google Sheets...")

for row in shipment_data:
    shipment_sheet.append_row(row)

print("Done! Supply chain report created.")
print("Supply Chain Sheet URL:", shipment_report.url)

Writing supply chain data to Google Sheets...
Done! Supply chain report created.
Supply Chain Sheet URL: https://docs.google.com/spreadsheets/d/1bT9ylLqA8bX7WjbwUYgBDOObKq7eEirF5bWtoOpLPq8


In [28]:
!pip install -q google-generativeai

## Gemini API Setup

In this section, I connect Google Colab to Gemini API.
I store the API key in Colab Secrets so the notebook stays secure.
Then I send datasets as prompts and ask Gemini to interpret them.

In [29]:
import os
from google.colab import userdata
from google import genai

# Pull the Gemini API key securely from Colab Secrets
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Initialize the Gemini client
client = genai.Client()

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [30]:
prompt_sales = """
Interpret this sales data set.
Format: [Amount, Region, Units]

sales_data = [
    [1200.00, 'E', 98.00],
    [5500.00, 'W', 355.00],
    [800.00, 'C', 16.00]
]

Please summarize the data and explain any patterns you notice.
"""

response_sales = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_sales
)

print(response_sales.text)

Let's break down this sales data.

## Sales Data Interpretation

**Dataset:**
*   **Region E (East):** $1200.00, 98.00 units
*   **Region W (West):** $5500.00, 355.00 units
*   **Region C (Central):** $800.00, 16.00 units

---

### Summary of Data

The dataset provides sales performance across three distinct geographical regions: East (E), West (W), and Central (C), measured by total sales amount and units sold.

*   **Total Sales Amount:** $1200.00 + $5500.00 + $800.00 = **$7500.00**
*   **Total Units Sold:** 98.00 + 355.00 + 16.00 = **469.00 units**

**Regional Breakdown:**

1.  **West (W) Region:**
    *   **Amount:** $5500.00
    *   **Units:** 355.00
    *   **Average Price per Unit (APU):** $5500 / 355 ≈ **$15.49**

2.  **East (E) Region:**
    *   **Amount:** $1200.00
    *   **Units:** 98.00
    *   **Average Price per Unit (APU):** $1200 / 98 ≈ **$12.24**

3.  **Central (C) Region:**
    *   **Amount:** $800.00
    *   **Units:** 16.00
    *   **Average Price per Unit (APU):**

In [31]:
prompt_supply_chain = """
Interpret this supply chain data set for a manager.
Format: [Order ID, Region, Delay Days]

shipment_data = [
    [1001, 'East', 2],
    [1002, 'West', 5],
    [1003, 'Central', 1]
]

Please explain which region has the largest delays and suggest simple actions
to improve delivery performance.
"""

response_supply_chain = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_supply_chain
)

print(response_supply_chain.text)

Here's an interpretation of the supply chain data for a manager, focusing on delays and actionable steps:

---

**Supply Chain Performance Update**

**Summary:**
Based on the provided data, the **West region** is currently experiencing the most significant delivery delays, with one order (ID 1002) delayed by 5 days. This is considerably higher than delays observed in the East (2 days) and Central (1 day) regions.

**Detailed Breakdown:**
*   **West Region:** Order 1002 was delayed by **5 days**. This is our primary concern based on this dataset.
*   **East Region:** Order 1001 was delayed by 2 days.
*   **Central Region:** Order 1003 was delayed by 1 day.

**Actionable Steps to Improve Delivery Performance:**

To address these delays, especially in the West region, I recommend the following simple actions:

1.  **Deep Dive into West Region Delays:**
    *   **Investigate Root Cause:** Immediately look into Order 1002 and any other West region shipments. Determine the specific cause of 

In [32]:
prompt_action = """
Using this shipment data:

shipment_data = [
    [1001, 'East', 2],
    [1002, 'West', 5],
    [1003, 'Central', 1]
]

Write a short manager-style summary with:
1. main issue
2. likely risk
3. one recommendation
"""

response_action = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_action
)

print(response_action.text)

Here's a short manager-style summary:

**Shipment Data Summary**

**1. Main Issue:**
The most significant issue is the extremely low shipment volume for the Central region, with only 1 package recorded compared to 2 for East and 5 for West.

**2. Likely Risk:**
This minimal volume in the Central region poses a risk of highly inefficient resource utilization, driving up the per-package delivery cost for that area, or potentially leading to customer dissatisfaction if shipments are delayed for consolidation.

**3. One Recommendation:**
I recommend investigating the root cause of this low volume in the Central region and exploring strategies for optimizing delivery logistics, such as consolidating shipments or adjusting delivery frequencies, to improve cost-effectiveness.
